# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library, referencing all entities by their `@id` fields for clarity and reproducibility.

### Dataset Source
The dataset is defined by a Croissant schema, accessible at:
<br>
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and preview the dataset with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"License: {getattr(metadata, 'license', 'Unknown')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'Unknown')}")

## 2. Data Overview
Review the available record sets and their fields by `@id`.

Below, we enumerate all record sets and for each, its associated fields as defined in the Croissant schema.

In [ ]:
# List all record sets by their @id
print("Record sets found in the dataset:")

record_sets = []
record_set_ids = []

# dataset.metadata.recordSet may be a list or dict
if hasattr(dataset.metadata, 'recordSet'):
    rs = dataset.metadata.recordSet
    if isinstance(rs, list):
        record_sets = rs
    elif rs is not None:
        record_sets = [rs]

# Print record sets and fields by @id
full_record_sets = []
for record_set in record_sets:
    rs_id = getattr(record_set, '@id', None)
    rs_name = getattr(record_set, 'name', None)
    print(f"- RecordSet @id: {rs_id}")
    if rs_name:
        print(f"  name: {rs_name}")
    record_set_ids.append(rs_id)
    # Fields in the record set
    fields = getattr(record_set, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        fld_id = getattr(field, '@id', str(field))
        fld_name = getattr(field, 'name', None)
        print(f"    - Field @id: {fld_id}" + (f" (name: {fld_name})" if fld_name else ''))
    print('')
    full_record_sets.append({'@id': rs_id, 'object': record_set, 'fields': fields})

if not record_set_ids:
    print("No record sets found in schema.\nTry to enumerate by accessing dataset.records().")

## 3. Data Extraction
Extract data from each record set to a pandas DataFrame.

- Use <b>`@id`</b> of each record set for reference.
- Enumerate fields (columns) of each DataFrame for clarity.

If no explicit record sets are present in the schema, use the dataset's default record set (the main CSV or tabular resource).

In [ ]:
# Prepare extraction
import collections

dataframes = collections.OrderedDict()
# If no explicit record sets, attempt with None ('default record set')
target_record_sets = record_set_ids if record_set_ids else [None]

for record_set_id in target_record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id or 'default'] = df
        print(f"  Fields (@id as columns):\n  {df.columns.tolist()}\n")
        print(df.head(2))
    else:
        print("  No records found.")

# For default progression, select the first (or only) available dataframe
main_record_set_id = list(dataframes.keys())[0]
main_df = dataframes[main_record_set_id]
print(f"\nMain record set for analysis: {main_record_set_id}\n")
print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Let's perform basic filtering, normalization, and groupings.

We will:
- Pick a numeric field (by inspecting DataFrame dtypes and column names, all referenced by `@id`)
- Filter for values above a threshold
- Normalize the numeric field
- Optionally group by a category field (by `@id`) if present

In [ ]:
# Find a numeric column by trying float/int types
numeric_candidates = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
if not numeric_candidates:
    # Attempt to infer numeric columns by trying to convert
    for col in main_df.columns:
        try:
            pd.to_numeric(main_df[col].dropna().iloc[:10])
            numeric_candidates.append(col)
        except Exception:
            continue
    
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field @id: {numeric_field_id}")
else:
    numeric_field_id = None
    print("No numeric field found in the main DataFrame.")

# Continue only if a numeric field is found
if numeric_field_id:
    df_num = main_df.copy()
    # Coerce to numeric (in case parsed as string)
    df_num[numeric_field_id] = pd.to_numeric(df_num[numeric_field_id], errors='coerce')

    threshold = df_num[numeric_field_id].dropna().quantile(0.5)  # median as example threshold
    filtered_df = df_num[df_num[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (50th percentile):")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to find a group-by candidate (non-numeric field)
    non_num_candidates = [col for col in main_df.columns if not pd.api.types.is_numeric_dtype(main_df[col])]
    if non_num_candidates:
        group_field = non_num_candidates[0]
        print(f"\nGrouping by field @id: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print('No suitable categorical group field found.')
else:
    print('Skipping EDA: no numeric column identified.')

## 5. Visualization
Now, plot a basic distribution of the selected numeric field, and optionally a grouped bar plot.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id:
    plt.figure(figsize=(7,4))
    main_df[numeric_field_id].hist(bins=20, edgecolor='k')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Grouped bar plot if group_field exists
    if 'group_field' in locals():
        top_groups = main_df[group_field].value_counts().index[:5]  # top 5
        subset = main_df[main_df[group_field].isin(top_groups)]
        plt.figure(figsize=(8,4))
        subset.groupby(group_field)[numeric_field_id].mean().plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
- We successfully loaded the ordered logistic regression dataset defined by a Croissant schema using `mlcroissant`.
- All record sets, fields, and columns are referenced by their `@id` for transparency and reproducibility.
- We explored the data structure, previewed records, filtered and normalized a numeric field, and visualized its distribution.

Continue to analyze further predictors, examine relationships, or apply statistical modeling as appropriate for your research context.